# X-ray 2D-3D Registration — Colab notebook (nanodrr)

This notebook builds the registration module using **nanodrr**, a pure-PyTorch
differentiable DRR renderer (no PyTorch3D dependency — installs cleanly
anywhere, including plain `pip install`, no source compilation needed).

It uses the **DeepFluoro** dataset (Grupp et al.), a real public benchmark for
2D/3D X-ray registration, downloaded automatically by the library.

**Before running**: `Runtime > Change runtime type` and select a GPU (T4) for faster rendering, though this also runs fine on CPU.

## 1. Install dependencies

`[all]` pulls in the optional plotting/animation utilities we'll use below.

In [ ]:
!pip install -q "nanodrr[all]" tqdm

import torch
print("torch:", torch.__version__, "| gpu available:", torch.cuda.is_available())

## 2. Load a subject and render a DRR (forward step)

Downloads the DeepFluoro CT volume the first time this runs (cached after).

In [ ]:
import matplotlib.pyplot as plt
import torch

from nanodrr.camera import make_k_inv, make_rt_inv
from nanodrr.data import Subject, download_deepfluoro
from nanodrr.drr import render
from nanodrr.plot import plot_drr

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

imagepath, labelpath = download_deepfluoro(subject_id=1)
subject = Subject.from_filepath(imagepath, labelpath).to(device)

# C-arm imaging parameters (source-to-detector distance, pixel spacing, image size)
sdd = 1020.0
delx = dely = 2.0
x0 = y0 = 0.0
height = width = 200

k_inv = make_k_inv(sdd, delx, dely, x0, y0, height, width, device=device)
sdd_t = torch.tensor([sdd], device=device)

rt_inv = make_rt_inv(
    torch.tensor([[0.0, 0.0, 0.0]], device=device),
    torch.tensor([[0.0, 850.0, 0.0]], device=device),
    orientation="AP",
    isocenter=subject.isocenter,
)

img = render(subject, k_inv, rt_inv, sdd_t, height, width)
plot_drr(img.sum(dim=1, keepdim=True), ticks=False)
plt.show()

## 3. Classical registration by gradient descent (inverse step)

Start from a wrong pose guess ("moving" image), and recover the true pose
("fixed" image) by backpropagating a normalized cross-correlation loss
directly into the pose parameters.

In [ ]:
from tqdm import tqdm

subject_flat = Subject.from_filepath(imagepath).to(device)

# Ground-truth ("fixed") pose
rt_inv_true = make_rt_inv(
    rotation=torch.tensor([[0.0, 0.0, 0.0]]),
    translation=torch.tensor([[0.0, 850.0, 0.0]]),
    orientation="AP",
    isocenter=subject_flat.isocenter.cpu(),
).to(dtype=torch.float32, device=device)

# Deliberately wrong starting ("moving") pose
rot_init = torch.tensor([[-0.1303, 0.3461, -0.7852]]) / torch.pi * 180
xyz_init = torch.tensor([[-11.8600, 828.8053, -24.4597]])
rt_inv_pred = make_rt_inv(
    rotation=rot_init,
    translation=xyz_init,
    orientation="AP",
    isocenter=subject_flat.isocenter.cpu(),
).to(dtype=torch.float32, device=device)

true_img = render(subject_flat, k_inv, rt_inv_true, sdd_t, height, width).sum(dim=1, keepdim=True)
moving_img = render(subject_flat, k_inv, rt_inv_pred, sdd_t, height, width).sum(dim=1, keepdim=True)

plot_drr(torch.concat([true_img, moving_img]), title=["Fixed (target)", "Moving (initial guess)"])
plt.show()

In [ ]:
from nanodrr.metrics import NormalizedCrossCorrelation2d
from nanodrr.registration import Registration

LR_ROT = 5e-2
LR_XYZ = 1e1
N_ITRS = 500
CONVERGENCE = 0.999

reg = Registration(subject_flat, rt_inv_pred, k_inv, sdd_t, height, width)
opt = torch.optim.Adam(
    [
        {"params": [reg._rot], "lr": LR_ROT},
        {"params": [reg._xyz], "lr": LR_XYZ},
    ],
    maximize=True,
)

ncc = NormalizedCrossCorrelation2d()

losses = []
poses = []
for idx in tqdm(range(N_ITRS)):
    opt.zero_grad()
    pred = reg()
    loss = ncc(true_img, pred)
    loss.backward()
    opt.step()
    losses.append(loss.item())
    poses.append(reg.pose.cpu().detach())
    if loss > CONVERGENCE:
        break

plt.plot(losses)
plt.xlabel("iteration")
plt.ylabel("normalized cross-correlation")
plt.title("2D-3D registration convergence")
plt.savefig("registration_convergence.png", dpi=150, bbox_inches="tight")
plt.show()

poses = torch.concat(poses).to(device)
print(f"Converged after {len(losses)} iterations, final NCC = {losses[-1]:.4f}")

## 4. Side-by-side comparison and animation

Visual sanity check of the recovered pose, plus an animated GIF of the
optimization trajectory — a strong visual for the README/demo.

In [ ]:
with torch.no_grad():
    recovered_img = render(subject_flat, k_inv, poses[-1:], sdd_t, height, width).sum(dim=1, keepdim=True)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(true_img.squeeze().cpu().numpy(), cmap="gray")
axes[0].set_title("Target (fixed)")
axes[0].axis("off")
axes[1].imshow(recovered_img.squeeze().cpu().numpy(), cmap="gray")
axes[1].set_title("Recovered pose")
axes[1].axis("off")
plt.tight_layout()
plt.savefig("registration_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
from nanodrr.plot import animate

moving_seq = render(subject_flat, k_inv, poses, sdd_t, height, width)
animate(moving_seq, fixed_img=true_img, canny_low=20)
# Saves/displays an animated GIF of the pose converging toward the target —
# download this from Colab's file browser for the README/demo.

## Next steps

- Download `registration_convergence.png`, `registration_comparison.png`, and the animation from Colab's file browser and add them to `results/` in the GitHub repo
- Once this baseline works, add a learned pose-prediction model for comparison (classical vs. learned)
- Move on to the segmentation module (ARCADE dataset)